# 05 — Raster Alignment and Quality Control (Fixed)

This notebook aligns all precipitation and predictor rasters to the first NDVI raster, fixes invalid Engineering CRS metadata for CDR/PDIR, and creates final QC reports.


In [2]:
from __future__ import annotations

import os
import sys
from datetime import datetime
from pathlib import Path
from typing import Optional

import numpy as np
import pandas as pd

# ----------------------------------------------------------
# PROJ / GDAL paths
# ----------------------------------------------------------

ENV_ROOT = Path(sys.prefix)
PROJ_DIR = ENV_ROOT / "Library" / "share" / "proj"
GDAL_DIR = ENV_ROOT / "Library" / "share" / "gdal"

if PROJ_DIR.exists():
    os.environ["PROJ_LIB"] = str(PROJ_DIR)
    os.environ["PROJ_DATA"] = str(PROJ_DIR)

if GDAL_DIR.exists():
    os.environ["GDAL_DATA"] = str(GDAL_DIR)

import rasterio
from rasterio.crs import CRS
from rasterio.warp import reproject, Resampling


# ----------------------------------------------------------
# Project root
# ----------------------------------------------------------

def find_project_root(start: Optional[Path] = None) -> Path:
    current = (start or Path.cwd()).resolve()

    for candidate in [current, *current.parents]:
        if (
            (candidate / "data" / "raw").exists()
            and (candidate / "data" / "processed").exists()
        ):
            return candidate

    raise FileNotFoundError(
        "Project root পাওয়া যায়নি। Repository-এর ভিতর থেকে notebook run করো।"
    )


PROJECT_ROOT = find_project_root()

RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

PRECIPITATION_DIR = RAW_DIR / "precipitation"
PREDICTOR_DIR = RAW_DIR / "predictors"

ALIGNED_ROOT = PROCESSED_DIR / "aligned_rasters"
ALIGNED_ROOT.mkdir(parents=True, exist_ok=True)

SKIP_ALREADY_VALID_OUTPUTS = True
OVERWRITE_INVALID_OUTPUTS = True

DESTINATION_NODATA = -9999.0
OUTPUT_DTYPE = "float32"
OUTPUT_COMPRESSION = "LZW"


# ----------------------------------------------------------
# Literal WGS 84 CRS
# Avoids CRS.from_epsg(4326)
# ----------------------------------------------------------

WGS84_WKT = """
GEOGCS["WGS 84",
    DATUM["WGS_1984",
        SPHEROID["WGS 84",6378137,298.257223563]],
    PRIMEM["Greenwich",0],
    UNIT["degree",0.0174532925199433],
    AXIS["Latitude",NORTH],
    AXIS["Longitude",EAST]]
""".strip()

WGS84_CRS = CRS.from_wkt(WGS84_WKT)

FORCE_WGS84_IF_INVALID = {
    "cdr",
    "persiann_cdr",
    "pdir",
    "pdir_now",
}

DEM_DERIVED_DATASETS = {
    "slope",
    "aspect",
    "distance_sea",
}

CATEGORICAL_DATASETS = {
    "aspect",
    "lulc",
    "landcover",
    "land_cover",
}


# ----------------------------------------------------------
# Helper functions
# ----------------------------------------------------------

def list_rasters(folder: Path) -> list[Path]:
    files = []

    for pattern in ("*.tif", "*.tiff", "*.TIF", "*.TIFF"):
        files.extend(folder.rglob(pattern))

    return sorted(set(files))


def dataset_name(path: Path) -> str:
    return path.parent.name.strip().lower()


def crs_is_invalid(crs) -> bool:
    if crs is None:
        return True

    try:
        text = crs.to_wkt().lower()
    except Exception:
        text = str(crs).lower()

    invalid_terms = (
        "engineeringcrs",
        "engineering_crs",
        "engineering datum",
        "unknown engineering",
        "local_cs",
    )

    return any(term in text for term in invalid_terms)


def bounds_are_geographic(bounds) -> bool:
    return (
        -180.0 <= bounds.left <= 180.0
        and -180.0 <= bounds.right <= 180.0
        and -90.0 <= bounds.bottom <= 90.0
        and -90.0 <= bounds.top <= 90.0
    )


def choose_resampling(path: Path) -> Resampling:
    if dataset_name(path) in CATEGORICAL_DATASETS:
        return Resampling.nearest

    return Resampling.bilinear


def safe_write_csv(dataframe: pd.DataFrame, path: Path) -> Path:
    path.parent.mkdir(parents=True, exist_ok=True)

    try:
        dataframe.to_csv(path, index=False, encoding="utf-8-sig")
        return path

    except PermissionError:
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        alternative = path.with_name(
            f"{path.stem}_{timestamp}{path.suffix}"
        )

        dataframe.to_csv(
            alternative,
            index=False,
            encoding="utf-8-sig",
        )

        return alternative


# ----------------------------------------------------------
# Reference raster
# ----------------------------------------------------------

reference_candidates = list_rasters(PREDICTOR_DIR / "NDVI")

if not reference_candidates:
    reference_candidates = list_rasters(PREDICTOR_DIR)

if not reference_candidates:
    raise FileNotFoundError(
        "data/raw/predictors-এর মধ্যে reference raster পাওয়া যায়নি।"
    )

REFERENCE_PATH = reference_candidates[0]

with rasterio.open(REFERENCE_PATH) as reference:
    if reference.crs is None or crs_is_invalid(reference.crs):
        raise ValueError(
            f"Reference raster-এর valid CRS নেই:\n{REFERENCE_PATH}"
        )

    REF_CRS = reference.crs
    REF_TRANSFORM = reference.transform
    REF_WIDTH = reference.width
    REF_HEIGHT = reference.height
    REF_RESOLUTION = reference.res
    REF_BOUNDS = reference.bounds
    REF_PROFILE = reference.profile.copy()


# ----------------------------------------------------------
# Valid DEM CRS
# ----------------------------------------------------------

DEM_CRS = None

for dem_path in list_rasters(PREDICTOR_DIR / "DEM"):
    try:
        with rasterio.open(dem_path) as dem:
            if dem.crs is not None and not crs_is_invalid(dem.crs):
                DEM_CRS = dem.crs
                break
    except Exception:
        continue


# ----------------------------------------------------------
# CRS handling
# ----------------------------------------------------------

def resolve_source_crs(source_path: Path, src):
    name = dataset_name(source_path)

    if src.crs is not None and not crs_is_invalid(src.crs):
        return src.crs, "original_crs"

    if name in FORCE_WGS84_IF_INVALID:
        return WGS84_CRS, "forced_wgs84_wkt"

    if bounds_are_geographic(src.bounds):
        return WGS84_CRS, "wgs84_wkt_from_bounds"

    if name in DEM_DERIVED_DATASETS and DEM_CRS is not None:
        return DEM_CRS, "assigned_dem_crs"

    raise ValueError(
        "CRS safely determine করা যায়নি.\n"
        f"File: {source_path}\n"
        f"Original CRS: {src.crs}\n"
        f"Bounds: {src.bounds}"
    )


def output_is_valid(path: Path) -> bool:
    if not path.exists():
        return False

    try:
        with rasterio.open(path) as src:
            return all([
                src.crs == REF_CRS,
                src.width == REF_WIDTH,
                src.height == REF_HEIGHT,
                src.transform == REF_TRANSFORM,
                src.bounds == REF_BOUNDS,
                src.res == REF_RESOLUTION,
                src.count == 1,
            ])
    except Exception:
        return False


# ----------------------------------------------------------
# Align one raster
# ----------------------------------------------------------

def align_one_raster(source_path: Path, output_path: Path) -> dict:
    if SKIP_ALREADY_VALID_OUTPUTS and output_is_valid(output_path):
        return {
            "source_file": str(source_path.relative_to(PROJECT_ROOT)),
            "output_file": str(output_path.relative_to(PROJECT_ROOT)),
            "dataset": source_path.parent.name,
            "status": "SKIPPED_VALID",
            "crs_action": "existing_valid_output",
            "source_crs_original": "",
            "source_crs_used": str(REF_CRS),
            "resampling": "",
            "valid_pixels": None,
            "total_pixels": REF_WIDTH * REF_HEIGHT,
            "valid_percentage": None,
            "error": "",
        }

    if output_path.exists() and not OVERWRITE_INVALID_OUTPUTS:
        return {
            "source_file": str(source_path.relative_to(PROJECT_ROOT)),
            "output_file": str(output_path.relative_to(PROJECT_ROOT)),
            "dataset": source_path.parent.name,
            "status": "SKIPPED_EXISTS",
            "crs_action": "",
            "source_crs_original": "",
            "source_crs_used": "",
            "resampling": "",
            "valid_pixels": None,
            "total_pixels": REF_WIDTH * REF_HEIGHT,
            "valid_percentage": None,
            "error": "",
        }

    output_path.parent.mkdir(parents=True, exist_ok=True)

    with rasterio.open(source_path) as src:
        source_crs, crs_action = resolve_source_crs(
            source_path,
            src,
        )

        source_array = src.read(1)
        source_nodata = src.nodata

        if source_nodata is None:
            if np.issubdtype(source_array.dtype, np.floating):
                source_array = np.where(
                    np.isfinite(source_array),
                    source_array,
                    np.nan,
                )
                source_nodata = np.nan

        destination = np.full(
            (REF_HEIGHT, REF_WIDTH),
            DESTINATION_NODATA,
            dtype=np.float32,
        )

        resampling_method = choose_resampling(source_path)

        reproject(
            source=source_array,
            destination=destination,
            src_transform=src.transform,
            src_crs=source_crs,
            src_nodata=source_nodata,
            dst_transform=REF_TRANSFORM,
            dst_crs=REF_CRS,
            dst_nodata=DESTINATION_NODATA,
            resampling=resampling_method,
            init_dest_nodata=True,
            num_threads=2,
        )

        profile = REF_PROFILE.copy()
        profile.update(
            driver="GTiff",
            width=REF_WIDTH,
            height=REF_HEIGHT,
            count=1,
            dtype=OUTPUT_DTYPE,
            crs=REF_CRS,
            transform=REF_TRANSFORM,
            nodata=DESTINATION_NODATA,
            compress=OUTPUT_COMPRESSION,
        )

        with rasterio.open(output_path, "w", **profile) as dst:
            dst.write(destination.astype(np.float32), 1)
            dst.set_band_description(1, source_path.stem)

        valid_mask = (
            np.isfinite(destination)
            & (destination != DESTINATION_NODATA)
        )

        valid_pixels = int(valid_mask.sum())
        total_pixels = int(destination.size)

        valid_percentage = (
            valid_pixels / total_pixels * 100.0
            if total_pixels
            else 0.0
        )

        return {
            "source_file": str(source_path.relative_to(PROJECT_ROOT)),
            "output_file": str(output_path.relative_to(PROJECT_ROOT)),
            "dataset": source_path.parent.name,
            "status": "SUCCESS",
            "crs_action": crs_action,
            "source_crs_original": str(src.crs),
            "source_crs_used": str(source_crs),
            "resampling": resampling_method.name,
            "valid_pixels": valid_pixels,
            "total_pixels": total_pixels,
            "valid_percentage": round(valid_percentage, 4),
            "error": "",
        }


# ----------------------------------------------------------
# Process all rasters
# ----------------------------------------------------------

all_rasters = sorted(set(
    list_rasters(PRECIPITATION_DIR)
    + list_rasters(PREDICTOR_DIR)
))

if not all_rasters:
    raise FileNotFoundError("কোনো raster পাওয়া যায়নি।")

print("=" * 72)
print("COMPLETE RASTER ALIGNMENT")
print("=" * 72)
print("Project root       :", PROJECT_ROOT)
print("Reference raster   :", REFERENCE_PATH)
print("Reference CRS      :", REF_CRS)
print("Reference size     :", REF_WIDTH, "x", REF_HEIGHT)
print("Total input rasters:", len(all_rasters))
print("=" * 72)

records = []

for index, source_path in enumerate(all_rasters, start=1):
    relative_path = source_path.relative_to(RAW_DIR)
    output_path = ALIGNED_ROOT / relative_path

    print(f"[{index:03d}/{len(all_rasters):03d}] {relative_path}")

    try:
        result = align_one_raster(source_path, output_path)
        records.append(result)

        print(
            f"    {result['status']} | "
            f"{result['crs_action']}"
        )

    except Exception as error:
        records.append({
            "source_file": str(source_path.relative_to(PROJECT_ROOT)),
            "output_file": str(output_path.relative_to(PROJECT_ROOT)),
            "dataset": source_path.parent.name,
            "status": "FAILED",
            "crs_action": "",
            "source_crs_original": "",
            "source_crs_used": "",
            "resampling": "",
            "valid_pixels": None,
            "total_pixels": REF_WIDTH * REF_HEIGHT,
            "valid_percentage": None,
            "error": str(error),
        })

        print("    FAILED:", error)


alignment_report = pd.DataFrame(records)

alignment_report_path = safe_write_csv(
    alignment_report,
    PROCESSED_DIR / "raster_alignment_report.csv",
)


# ----------------------------------------------------------
# Final QC
# ----------------------------------------------------------

qc_records = []
aligned_files = list_rasters(ALIGNED_ROOT)

for aligned_path in aligned_files:
    try:
        with rasterio.open(aligned_path) as src:
            crs_match = src.crs == REF_CRS
            shape_match = (
                src.width == REF_WIDTH
                and src.height == REF_HEIGHT
            )
            resolution_match = src.res == REF_RESOLUTION
            transform_match = src.transform == REF_TRANSFORM
            bounds_match = src.bounds == REF_BOUNDS

            passed = all([
                crs_match,
                shape_match,
                resolution_match,
                transform_match,
                bounds_match,
            ])

            qc_records.append({
                "file": str(aligned_path.relative_to(PROJECT_ROOT)),
                "dataset": aligned_path.parent.name,
                "crs_match": crs_match,
                "shape_match": shape_match,
                "resolution_match": resolution_match,
                "transform_match": transform_match,
                "bounds_match": bounds_match,
                "qc_status": "PASS" if passed else "FAIL",
                "error": "",
            })

    except Exception as error:
        qc_records.append({
            "file": str(aligned_path.relative_to(PROJECT_ROOT)),
            "dataset": aligned_path.parent.name,
            "crs_match": False,
            "shape_match": False,
            "resolution_match": False,
            "transform_match": False,
            "bounds_match": False,
            "qc_status": "ERROR",
            "error": str(error),
        })


qc_report = pd.DataFrame(qc_records)

qc_report_path = safe_write_csv(
    qc_report,
    PROCESSED_DIR / "final_raster_alignment_qc.csv",
)

dataset_summary = (
    alignment_report
    .groupby(["dataset", "status"], dropna=False)
    .size()
    .reset_index(name="count")
    .sort_values(["dataset", "status"])
)

dataset_summary_path = safe_write_csv(
    dataset_summary,
    PROCESSED_DIR / "raster_alignment_dataset_summary.csv",
)


# ----------------------------------------------------------
# Final output
# ----------------------------------------------------------

print("\n" + "=" * 72)
print("ALIGNMENT SUMMARY")
print("=" * 72)
print(alignment_report["status"].value_counts(dropna=False))

print("\n" + "=" * 72)
print("FINAL QC SUMMARY")
print("=" * 72)

if qc_report.empty:
    print("No aligned outputs found.")
else:
    print(qc_report["qc_status"].value_counts(dropna=False))

print("\nTotal input rasters :", len(all_rasters))
print("Total aligned files :", len(aligned_files))

print("\nReports:")
print("1.", alignment_report_path)
print("2.", qc_report_path)
print("3.", dataset_summary_path)

failed = alignment_report[
    alignment_report["status"] == "FAILED"
]

print("\nFailed rasters:", len(failed))

if not failed.empty:
    display(
        failed[
            ["source_file", "dataset", "error"]
        ]
    )
else:
    print("None")

display(dataset_summary)

COMPLETE RASTER ALIGNMENT
Project root       : E:\Geospatial\Precipitation-Downscaling-Khulna\Machine-Learning-Based-Precipitation-Downscaling-over-Khulna-District-Bangladesh
Reference raster   : E:\Geospatial\Precipitation-Downscaling-Khulna\Machine-Learning-Based-Precipitation-Downscaling-over-Khulna-District-Bangladesh\data\raw\predictors\NDVI\NDVI_2017_1.tif
Reference CRS      : GEOGCS["WGS 84",DATUM["World Geodetic System 1984",SPHEROID["WGS 84",6378137,298.257223563]],PRIMEM["Greenwich",0],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]],AXIS["Latitude",NORTH],AXIS["Longitude",EAST]]
Reference size     : 59 x 151
Total input rasters: 800
[001/800] precipitation\CCS\2017_01.tif
    SKIPPED_VALID | existing_valid_output
[002/800] precipitation\CCS\2017_02.tif
    SKIPPED_VALID | existing_valid_output
[003/800] precipitation\CCS\2017_03.tif
    SKIPPED_VALID | existing_valid_output
[004/800] precipitation\CCS\2017_04.tif
    SKIPPED_VALID | existing_valid_output
[005/800] 

,dataset,status,count
0,Aspect,SKIPPED_VALID,1
1,CCS,SKIPPED_VALID,72
2,CDR,SKIPPED_VALID,72
3,CHIRPS_TIFF_2017_2022,SKIPPED_VALID,72
4,DEM,SKIPPED_VALID,1
5,Distance_Sea,SKIPPED_VALID,5
6,ERA5_TIFF,SKIPPED_VALID,72
7,GSMaP_Gauge_v7,SKIPPED_VALID,72
8,GSMaP_MVK,SKIPPED_VALID,72
9,IMERG_Monthly,SKIPPED_VALID,72


## Expected final result

```text
Total input rasters : 800
Total aligned files : 800
PASS : 800
Failed rasters: 0
```
